# Citi Bike Jersey City: Member vs Casual Trip Classifier

**Purpose of this notebook:** a single, end-to-end, runnable notebook (pandas + scikit-learn) that the team will "destroy" into modules in Sessions 4 to 9. Each section maps to one module in `1-mlops-project-skeleton`:

| Section | Skeleton module |
|---|---|
| 1. Settings | `main.py` (SETTINGS) |
| 2. Load data | `load_data.py` |
| 3. Validate raw data | `validate.py` |
| 4. Clean data | `clean_data.py` |
| 5. Trip features | `features.py` |
| 6. Train/test split | `main.py` / `train.py` |
| 7. Station features (fit on train only) | `features.py` |
| 8. Train model | `train.py` |
| 9. Evaluate | `evaluate.py` |
| 10. Save artifacts | `utils.py` |
| 11. Predict and find conversion targets | `infer.py` |

**ML problem:** predict whether a trip was made by a **member** (1) or a **casual** rider (0). Casual trips the model scores as very member-like are candidates for a membership offer.

**Data:** Citi Bike Jersey City and Hoboken trip data, August and September 2026 (217,855 trips), from the public Citi Bike System Data. Place the two monthly CSVs in `data/raw/`. Data files are kept out of Git.

**Changes from the original MDA2 Spark notebook**
- Spark, GraphFrames and MinIO replaced by pandas, networkx and scikit-learn, so it runs on a laptop with one environment
- No credentials in the code
- One ML problem (Q5). The descriptive graph and clustering analysis (Q1 to Q4) is left out of the pipeline
- **Leakage fixed:** station features are now learned from the training set only, and the label-derived `casual_share` station feature is no longer used

## 1. Settings
Every critical value lives here, so later it can move to `main.py` (and in stage 3 to `config.yaml`) without hunting through cells.

In [1]:
from pathlib import Path

import joblib
import networkx as nx
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Repo root works whether the notebook runs from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

SETTINGS = {
    "raw_glob": "JC-*-citibike-tripdata.csv",
    "raw_dir": ROOT / "data" / "raw",
    "clean_path": ROOT / "data" / "processed" / "clean.csv",
    "model_path": ROOT / "models" / "model.joblib",
    "predictions_path": ROOT / "reports" / "predictions.csv",
    "target": "member_casual",
    "positive_class": "member",
    "valid_station_prefixes": ("JC", "HB"),
    "min_duration_min": 1,
    "max_duration_min": 180,
    "test_size": 0.2,
    "random_state": 42,
    "conversion_threshold": 0.8,
    "categorical_features": ["rideable_type", "hour", "day_of_week"],
    "numeric_features": ["is_weekend", "duration_min", "is_round_trip",
                         "start_pagerank", "end_pagerank"],
}

REQUIRED_COLUMNS = [
    "ride_id", "rideable_type", "started_at", "ended_at",
    "start_station_name", "start_station_id", "end_station_name", "end_station_id",
    "start_lat", "start_lng", "end_lat", "end_lng", "member_casual",
]

## 2. Load data
Read every monthly file in `data/raw/` and stack them. Timestamps are parsed on load.

In [2]:
def load_raw_trips(raw_dir: Path, pattern: str) -> pd.DataFrame:
    files = sorted(raw_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern} in {raw_dir}")
    frames = [pd.read_csv(f, parse_dates=["started_at", "ended_at"],
                          dtype={"start_station_id": str, "end_station_id": str})
              for f in files]
    return pd.concat(frames, ignore_index=True)


raw_df = load_raw_trips(SETTINGS["raw_dir"], SETTINGS["raw_glob"])
print(f"Loaded {len(raw_df):,} trips")
raw_df.head()

Loaded 217,855 trips


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,838514AD69A213B0,electric_bike,2026-08-23 07:43:32.963,2026-08-23 07:47:53.366,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member
1,C558F0271F2D8A69,electric_bike,2026-08-27 18:47:17.159,2026-08-27 18:52:27.713,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,casual
2,3BEED20F1EA3C962,classic_bike,2026-08-18 21:11:22.555,2026-08-18 21:17:25.595,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member
3,588E3A9794C177C8,electric_bike,2026-08-27 07:00:47.836,2026-08-27 07:02:21.681,Madison St & 1 St,HB402,Marshall St & 2 St,HB408,40.738790,-74.039300,40.740802,-74.042521,member
4,3BD0D2BD8033E7D1,classic_bike,2026-08-23 12:39:13.361,2026-08-23 12:51:11.762,Jersey & 6th St,JC027,Marshall St & 2 St,HB408,40.725289,-74.045572,40.740802,-74.042521,member


## 3. Validate raw data
Fail fast with a clear error if the input does not match what the pipeline expects.

In [3]:
def validate_raw_trips(df: pd.DataFrame) -> None:
    missing = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {sorted(missing)}")
    if df.empty:
        raise ValueError("Input data is empty")
    if df["ride_id"].duplicated().any():
        raise ValueError("Duplicate ride_id values found")
    bad_labels = set(df["member_casual"].dropna().unique()) - {"member", "casual"}
    if bad_labels:
        raise ValueError(f"Unexpected member_casual values: {bad_labels}")
    if df["member_casual"].isna().any():
        raise ValueError("member_casual has missing values")


validate_raw_trips(raw_df)
print("Raw data passed validation")
print(raw_df.isna().sum()[lambda s: s > 0])
print(raw_df["member_casual"].value_counts(normalize=True).round(3))

Raw data passed validation
start_station_name      2
start_station_id        2
end_station_name      576
end_station_id        681
start_lat               2
start_lng               2
end_lat               681
end_lng               681
dtype: int64
member_casual
member    0.729
casual    0.271
Name: proportion, dtype: float64


## 4. Clean data
The same three rules as the original notebook, applied one at a time so each removal is visible:
1. Drop trips without a start or end station ID (undocked e-bikes)
2. Keep only trips where both stations are in Jersey City or Hoboken (`JC` or `HB` IDs)
3. Keep trips lasting 1 to 180 minutes

The input DataFrame is never modified.

In [4]:
def clean_trips(df: pd.DataFrame, prefixes: tuple, min_min: float, max_min: float) -> pd.DataFrame:
    out = df.copy()
    n0 = len(out)

    out = out.dropna(subset=["start_station_id", "end_station_id"])
    n1 = len(out)

    in_area = (out["start_station_id"].str.startswith(prefixes)
               & out["end_station_id"].str.startswith(prefixes))
    out = out[in_area]
    n2 = len(out)

    out["duration_min"] = (out["ended_at"] - out["started_at"]).dt.total_seconds() / 60
    out = out[out["duration_min"].between(min_min, max_min)]
    n3 = len(out)

    print(f"Raw trips:                 {n0:>8,}")
    print(f"After rule 1 (station IDs): {n1:>8,}  removed {n0 - n1:>5,}")
    print(f"After rule 2 (JC/HB only):  {n2:>8,}  removed {n1 - n2:>5,}")
    print(f"After rule 3 (1-180 min):   {n3:>8,}  removed {n2 - n3:>5,}")
    print(f"Kept: {n3 / n0:.2%}")
    return out.reset_index(drop=True)


clean_df = clean_trips(raw_df, SETTINGS["valid_station_prefixes"],
                       SETTINGS["min_duration_min"], SETTINGS["max_duration_min"])

Raw trips:                  217,855
After rule 1 (station IDs):  217,172  removed   683
After rule 2 (JC/HB only):   216,639  removed   533
After rule 3 (1-180 min):    216,422  removed   217
Kept: 99.34%


## 5. Trip features
Row-level features that use only the trip itself, so they cannot leak information between train and test:
- `hour` (0 to 23), `day_of_week` (0 = Monday to 6 = Sunday), `is_weekend`
- `is_round_trip`: starts and ends at the same station
- `label`: 1 = member, 0 = casual

In [5]:
def add_trip_features(df: pd.DataFrame, target: str, positive_class: str) -> pd.DataFrame:
    out = df.copy()
    out["hour"] = out["started_at"].dt.hour
    out["day_of_week"] = out["started_at"].dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["is_round_trip"] = (out["start_station_id"] == out["end_station_id"]).astype(int)
    out["label"] = (out[target] == positive_class).astype(int)
    return out


trips_df = add_trip_features(clean_df, SETTINGS["target"], SETTINGS["positive_class"])
print(trips_df.groupby("is_weekend")["label"].agg(trips="size", member_share="mean").round(3))

             trips  member_share
is_weekend                      
0           159938         0.761
1            56484         0.640


## 6. Train/test split
Stratified 80/20 split with a fixed seed, so the member/casual balance is the same in both sets and the split is identical on every laptop. **Everything learned from data happens after this cell, on the training set only.**

In [6]:
train_df, test_df = train_test_split(
    trips_df,
    test_size=SETTINGS["test_size"],
    stratify=trips_df["label"],
    random_state=SETTINGS["random_state"],
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, df in [("Train", train_df), ("Test", test_df)]:
    print(f"{name}: {len(df):>7,} trips, member share = {df['label'].mean():.1%}")

Train: 173,137 trips, member share = 73.0%
Test:  43,285 trips, member share = 73.0%


## 7. Station features (fit on train only)
This keeps the graph idea from the original notebook: stations are nodes and trips are directed, weighted edges. **PageRank** scores how central each station is.

The fix: PageRank is computed from **training trips only**, then looked up for test trips. A station that never appears in training gets the median score. The original `casual_share` station feature is dropped, because it was calculated from the label itself.

In [7]:
def fit_station_pagerank(df: pd.DataFrame) -> pd.Series:
    edges = df.groupby(["start_station_id", "end_station_id"]).size().reset_index(name="weight")
    graph = nx.DiGraph()
    graph.add_weighted_edges_from(edges.itertuples(index=False, name=None))
    scores = nx.pagerank(graph, alpha=0.85, weight="weight")
    # Rescale so the average station scores 1, as in GraphFrames
    scores = pd.Series(scores) * len(scores)
    return scores


def add_station_features(df: pd.DataFrame, pagerank: pd.Series) -> pd.DataFrame:
    out = df.copy()
    fallback = float(pagerank.median())
    out["start_pagerank"] = out["start_station_id"].map(pagerank).fillna(fallback)
    out["end_pagerank"] = out["end_station_id"].map(pagerank).fillna(fallback)
    return out


station_pagerank = fit_station_pagerank(train_df)
train_df = add_station_features(train_df, station_pagerank)
test_df = add_station_features(test_df, station_pagerank)

names = clean_df.drop_duplicates("start_station_id").set_index("start_station_id")["start_station_name"]
print("Most central stations (from training trips):")
print(station_pagerank.sort_values(ascending=False).head(5).rename(index=names).round(2))

Most central stations (from training trips):
Grove St PATH                   4.95
Bergen Ave & Sip Ave            3.37
River St & 1 St                 2.77
Exchange Pl                     2.45
Hudson Pl & Hoboken Terminal    2.30
dtype: float64


## 8. Train model
One scikit-learn `Pipeline`: one-hot encode the categorical columns, scale the numeric ones, then classify. Because preprocessing lives inside the pipeline, it is fitted on training data only and saved together with the model.

We compare a Logistic Regression baseline with Gradient Boosting (the scikit-learn counterpart of the Spark GBT that won in the original notebook).

In [8]:
def build_pipeline(model) -> Pipeline:
    preprocess = ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), SETTINGS["categorical_features"]),
        ("numeric", StandardScaler(), SETTINGS["numeric_features"]),
    ])
    return Pipeline([("preprocess", preprocess), ("model", model)])


FEATURES = SETTINGS["categorical_features"] + SETTINGS["numeric_features"]
X_train, y_train = train_df[FEATURES], train_df["label"]
X_test, y_test = test_df[FEATURES], test_df["label"]

candidates = {
    "logistic_regression": LogisticRegression(max_iter=1000),
    "gradient_boosting": HistGradientBoostingClassifier(random_state=SETTINGS["random_state"]),
}

fitted = {name: build_pipeline(m).fit(X_train, y_train) for name, m in candidates.items()}
print("Trained:", list(fitted))

Trained: ['logistic_regression', 'gradient_boosting']


## 9. Evaluate
**Metric: `roc_auc`** on the unseen test set. 73% of trips are by members, so a model that always answers "member" already reaches about 73% accuracy while learning nothing. ROC AUC measures how well the model ranks members above casual riders (0.5 = random, 1.0 = perfect), which is what we need to rank casual trips by how member-like they are. Accuracy is shown next to the majority-class baseline for reference.

In [9]:
def evaluate(pipeline: Pipeline, X: pd.DataFrame, y: pd.Series) -> dict:
    proba = pipeline.predict_proba(X)[:, 1]
    return {
        "roc_auc": round(roc_auc_score(y, proba), 4),
        "accuracy": round(accuracy_score(y, (proba >= 0.5).astype(int)), 4),
    }


results = pd.DataFrame({name: evaluate(p, X_test, y_test) for name, p in fitted.items()}).T
results.loc["majority_baseline"] = {"roc_auc": 0.5, "accuracy": round(y_test.mean(), 4)}
results = results.sort_values("roc_auc", ascending=False)
results

,roc_auc,accuracy
gradient_boosting,0.7286,0.7504
logistic_regression,0.6823,0.7404
majority_baseline,0.5000,0.7296


In [10]:
best_name = results.drop("majority_baseline")["roc_auc"].idxmax()
best_model = fitted[best_name]
print(f"Selected model: {best_name} (test roc_auc = {results.loc[best_name, 'roc_auc']})")

Selected model: gradient_boosting (test roc_auc = 0.7286)


## 10. Save artifacts
The three files the pipeline must produce: `clean.csv`, `model.joblib` and `predictions.csv`. The folders exist in the skeleton and are git-ignored.

In [11]:
for key in ["clean_path", "model_path", "predictions_path"]:
    SETTINGS[key].parent.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(SETTINGS["clean_path"], index=False)
joblib.dump({"pipeline": best_model, "station_pagerank": station_pagerank,
             "features": FEATURES}, SETTINGS["model_path"])
print("Saved:", SETTINGS["clean_path"].name, SETTINGS["model_path"].name)

Saved: clean.csv model.joblib


## 11. Predict and find conversion targets
`infer.py` loads the saved model and scores trips. Here we score the **casual trips in the test set** (never seen in training). `p_member` is the probability that a trip was made by a member; casual trips at or above the conversion threshold are ridden like member trips, so those riders are candidates for a membership offer.

In [12]:
def predict(artifact: dict, df: pd.DataFrame) -> pd.DataFrame:
    scored = add_station_features(df, artifact["station_pagerank"])
    out = df[["ride_id"]].copy()
    out["p_member"] = artifact["pipeline"].predict_proba(scored[artifact["features"]])[:, 1].round(3)
    return out


artifact = joblib.load(SETTINGS["model_path"])
test_casual = test_df[test_df["label"] == 0]
predictions = predict(artifact, test_casual)
predictions["conversion_target"] = (predictions["p_member"] >= SETTINGS["conversion_threshold"]).astype(int)
predictions.to_csv(SETTINGS["predictions_path"], index=False)

share = predictions["conversion_target"].mean()
print(f"Casual test trips scored: {len(predictions):,}")
print(f"Conversion targets (p_member >= {SETTINGS['conversion_threshold']}): "
      f"{predictions['conversion_target'].sum():,} ({share:.1%})")

Casual test trips scored: 11,704
Conversion targets (p_member >= 0.8): 1,697 (14.5%)


In [13]:
profile = test_casual.merge(predictions, on="ride_id")
profile["segment"] = np.where(profile["conversion_target"] == 1, "Conversion target", "Other casual")
profile["rush_hour"] = profile["hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
profile["electric"] = (profile["rideable_type"] == "electric_bike").astype(int)

profile.groupby("segment").agg(
    trips=("ride_id", "size"),
    weekday_share=("is_weekend", lambda s: 1 - s.mean()),
    rush_hour_share=("rush_hour", "mean"),
    avg_duration_min=("duration_min", "mean"),
    electric_share=("electric", "mean"),
).round(3)

,trips,weekday_share,rush_hour_share,avg_duration_min,electric_share
segment,,,,,
Conversion target,1697,0.872,0.511,4.963,0.468
Other casual,10007,0.608,0.348,14.356,0.704


## 12. Notes for the team
- **Next step:** the admin runs the kick-off scaffold helper with this notebook and the README business case as inputs
- **Limitations:** two summer months only; no rider ID, so targets are trips, not people; a single stratified split (a time-based split, train on August and test on September, would be a stricter test of production behaviour)
- **Not in the pipeline:** the descriptive network analysis from MDA2 (hubs, net flow, communities, station clustering). It stays in the original notebook as background for the business case